In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
import asyncio
from openai import AsyncOpenAI
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, Email, To, Content
from agents import set_default_openai_client, set_tracing_export_api_key, set_default_openai_api

In [2]:
load_dotenv(override=True)

True

In [3]:
anthropic_client = AsyncOpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url="https://api.anthropic.com/v1/")
set_default_openai_client(anthropic_client)
set_default_openai_api("chat_completions")
set_tracing_export_api_key(os.environ["OPENAI_API_KEY"])

In [4]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [6]:
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="claude-sonnet-4-6"
)
sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="claude-sonnet-4-6"
)
sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="claude-sonnet-4-6"
)

In [7]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here is a cold sales email:

---

**Subject: Is Your SOC2 Audit Keeping You Up at Night?**

---

Dear [First Name],

SOC2 compliance is no longer optional for companies handling sensitive customer data — it is a business imperative. Yet for many organizations, the audit preparation process remains a costly, time-consuming, and often chaotic undertaking.

That is where ComplAI comes in.

We provide an AI-powered compliance platform purpose-built to simplify SOC2 readiness. Our tool continuously monitors your controls, identifies gaps before auditors do, and generates the documentation you need — cutting audit preparation time by up to 60%.

Here is what that means for your organization:

- **Reduced risk** of audit failures or costly findings
- **Significant time savings** for your engineering and security teams
- **A clear, real-time view** of your compliance posture at all times

Companies that once spent months scrambling before an audit are now walking in with confidence.

I would w

In [86]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
outputs = [result.final_output for result in results]
for output in outputs:
    print(output + "\n\n")

CancelledError: 

In [7]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
        Imagine you are a customer and pick the one you are most likely to respond to. \
            Do not give an exprlanation; reply with the selected email only.",
    model="claude-sonnet-4-6",
)

In [90]:
with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]
    emails = "Cold sales emails:\n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(f"Best sales email:\n{best.final_output}")

CancelledError: 

In [8]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    try:
        sg = sendgrid.SendGridAPIClient(os.environ.get('SENDGRID_API_KEY'))
        from_email = Email("edengebeta210@gmail.com")
        to_email = To("eden.alemayehu@a2sv.org")
        content = Content("text/plain", body)
        mail = Mail(from_email, to_email, "Sales email", content).get()
        response = sg.client.mail.send.post(request_body=mail)
        print("STATUS:", response.status_code, flush=True)
        print("BODY:", response.body, flush=True)
        return {"status": "success", "code": response.status_code}
    except Exception as e:
        err_body = getattr(e, "body", str(e))
        print("SENDGRID ERROR:", err_body, flush=True)
        return {"status": "error", "detail": str(err_body)}


In [9]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001DE7EED7390>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [10]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool1

FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001DE7ECCB5C0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [11]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001DE7EF9E570>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties'

In [13]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="claude-sonnet-4-6")


In [62]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

STATUS: 202
BODY: b''
✅ **Email sent successfully!**

Here's a quick summary of my decision-making:

| Draft | Tone | Strength | Weakness |
|-------|------|----------|----------|
| Writer 1 | Formal & authoritative | Great business case | Too long for cold outreach |
| Writer 2 | Humorous & conversational | Very engaging, memorable | Tone may alienate some prospects |
| **Writer 3** ✅ | **Concise & data-driven** | **Short, clear, credible metrics** | **Minimal — best overall** |

**Why Draft 3 won:** In cold sales emails, attention is the scarcest resource. Draft 3 respects the reader's time, leads with a pain point, delivers specific proof points (80% reduction, zero surprise findings), and closes with a frictionless CTA. It's the email most likely to get a reply.


In [14]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."
html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."
subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="claude-sonnet-4-6")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")
html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="claude-sonnet-4-6")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [15]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    try:
        """ Send out an email with the given subject and HTML body to all sales prospects """
        sg = sendgrid.SendGridAPIClient(os.environ.get("SENDGRID_API_KEY"))
        from_email = Email("edengebeta210@gmail.com")  # Change to your verified sender
        to_email = To("eden.alemayehu@a2sv.org")       # Change to your recipient
        content = Content("text/html", html_body)
        mail = Mail(from_email, to_email, subject, content).get()
        response = sg.client.mail.send.post(request_body=mail)
        print("STATUS:", response.status_code, flush=True)
        print("BODY:", response.body, flush=True)
        return {"status": "success", "code": response.status_code}
    except Exception as e:
        err_body = getattr(e, "body", str(e))
        print("SENDGRID ERROR:", err_body, flush=True)
        return {"status": "error", "detail": str(err_body)}

In [16]:
tools = [subject_tool, html_tool, send_html_email]

In [19]:
instructions = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="claude-sonnet-4-6",
    handoff_description="Convert an email to HTML and send it")

In [20]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001DE7EF9E570>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties':

In [21]:
sales_manager_instructions = """You are a sales manager working for ComPLAI. You use the tools given to you to generate cold sales emails. 
You never generate sales emails yourself; you always use the tools. 
You try all 3 sales agent tools at least once before choosing the best one. 
You can use the tools multiple times. If you are not satisfied with the results from the first try, 
You select the single best email using your own judgement of which email will be most effective. 
After picking the email, you handoff to the Email Manager agent to format and send the email."""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="claude-sonnet-4-6")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

STATUS: 202
BODY: b''
